In [2]:
!pip install wbgapi

In [ ]:
import matplotlib.pyplot as plt

for group in bank_by_income['income_group'].unique():
    subset = bank_by_income[bank_by_income['income_group'] == group]
    plt.plot(subset['time'], subset['bank_account_access_pct'], marker='o', label=group)

plt.title('Bank Account Access by Income Group (2010-2024)')
plt.xlabel('Year')
plt.ylabel('% of Adults with Bank Account')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import wbgapi as wb
import pandas as pd

# Bank account access by income group 2010-2024
bank_by_income = wb.data.DataFrame(
    'FX.OWN.TOTL.ZS',
    economy=['WB_HI', 'WB_UMC', 'WB_LMC', 'WB_LI'],
    time=range(2010, 2025),
    labels=True
).reset_index()

bank_by_income = bank_by_income.rename(columns={
    'economy': 'income_group',
    'FX.OWN.TOTL.ZS': 'bank_account_access_pct'
})

# Clean up income group names
income_name_map = {
    'WB_HI' : 'High Income',
    'WB_UMC': 'Upper Middle Income',
    'WB_LMC': 'Lower Middle Income',
    'WB_LI' : 'Low Income'
}
bank_by_income['income_group'] = bank_by_income['income_group'].replace(income_name_map)

print(f'Shape: {bank_by_income.shape}')
bank_by_income

In [2]:
!pip install wbgapi

In [4]:
import wbgapi as wb
import pandas as pd

indicators = {
    'NY.GDP.PCAP.CD' : 'gdp_per_capita',
    'SL.UEM.1524.ZS' : 'youth_unemployment_pct',
    'FP.CPI.TOTL.ZG' : 'inflation_rate',
    'FX.OWN.TOTL.ZS' : 'bank_account_access_pct',
    'IT.NET.USER.ZS' : 'internet_users_pct'
}

wb_data = wb.data.DataFrame(
    list(indicators.keys()),
    time=2022,
    labels=True
).reset_index()

wb_data = wb_data.rename(columns=indicators)
wb_data = wb_data.rename(columns={'economy': 'country'})

print(f'Shape: {wb_data.shape}')
wb_data.head()

Shape: (266, 7)


,country,Country,inflation_rate,bank_account_access_pct,internet_users_pct,gdp_per_capita,youth_unemployment_pct
0,ZWE,Zimbabwe,104.705171,NaN,37.8861,2536.400502,16.984
1,ZMB,Zambia,10.993204,NaN,15.0000,1447.123101,9.849
2,YEM,"Yemen, Rep.",NaN,11.903366,NaN,NaN,32.454
3,PSE,West Bank and Gaza,3.741224,NaN,88.6469,3799.955270,36.092
4,VIR,Virgin Islands (U.S.),NaN,NaN,NaN,44320.909186,25.611


In [10]:

wb_data = wb_data.drop(columns=['country'])
wb_data = wb_data.rename(columns={'Country': 'country'})

wb_data = wb_data.dropna(subset=['gdp_per_capita', 'youth_unemployment_pct'])

print(f'Shape after cleaning: {wb_data.shape}')
wb_data.head()

Shape after cleaning: (229, 5)


,inflation_rate,bank_account_access_pct,internet_users_pct,gdp_per_capita,youth_unemployment_pct
0,104.705171,NaN,37.8861,2536.400502,16.984
1,10.993204,NaN,15.0000,1447.123101,9.849
3,3.741224,NaN,88.6469,3799.955270,36.092
4,NaN,NaN,NaN,44320.909186,25.611
5,3.156507,56.265754,78.5900,4147.697772,6.097


In [12]:
crypto_df = pd.read_csv('/Users/hei0613/Desktop/Project-DSCI-100/data/raw/crypto_adoption_2024.csv')
print(f'Shape: {crypto_df.shape}')
crypto_df.head()

Shape: (151, 7)


,country,region,overall_rank_2024,sub1_centralized_value,sub2_retail_centralized,sub3_defi_value,sub4_retail_defi
0,India,Central & Southern Asia and Oceania,1,1,1,3,2
1,Nigeria,Sub-Saharan Africa,2,5,2,2,3
2,Indonesia,Central & Southern Asia and Oceania,3,6,6,1,1
3,United States,North America,4,2,12,4,4
4,Vietnam,Central & Southern Asia and Oceania,5,3,3,6,5


In [23]:
# Clean column names
crypto_df.columns = crypto_df.columns.str.strip()
wb_data.columns = wb_data.columns.str.strip()

print("crypto_df columns:", crypto_df.columns.tolist())
print("wb_data columns:", wb_data.columns.tolist())

crypto_df columns: ['country', 'region', 'overall_rank_2024', 'sub1_centralized_value', 'sub2_retail_centralized', 'sub3_defi_value', 'sub4_retail_defi']
wb_data columns: ['inflation_rate', 'bank_account_access_pct', 'internet_users_pct', 'gdp_per_capita', 'youth_unemployment_pct']


In [24]:
# Merge crypto + World Bank data
master_df = crypto_df.merge(
    wb_data,
    left_on='country',
    right_on='country',
    how='left'
)

# Convert rank to score
master_df['adoption_score'] = 152 - master_df['overall_rank_2024']

# Create High/Medium/Low category
master_df['adoption_category'] = pd.cut(
    master_df['overall_rank_2024'],
    bins=[0, 50, 100, 151],
    labels=['High', 'Medium', 'Low']
)

print(f'Shape: {master_df.shape}')
print(f'\nMissing values:\n{master_df.isnull().sum()}')
master_df.head()

KeyError: 'country'